# 02 - Delta Lake

Este notebook implementa uma tabela `Delta Lake` local com base no arquivo `data/vendas.csv`. O foco e demonstrar as operacoes pedidas no trabalho: `INSERT`, `UPDATE` e `DELETE`.


In [ ]:
import os
from pathlib import Path
import shutil

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession, functions as F

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

LOCAL_HADOOP = PROJECT_ROOT / "windows-hadoop"
WINUTILS = LOCAL_HADOOP / "bin" / "winutils.exe"
if WINUTILS.exists():
    os.environ["HADOOP_HOME"] = str(LOCAL_HADOOP)
    os.environ["hadoop.home.dir"] = str(LOCAL_HADOOP)
    os.environ["PATH"] = f"{LOCAL_HADOOP / 'bin'};" + os.environ.get("PATH", "")
else:
    print("Aviso: winutils.exe nao encontrado. Em Windows, coloque o arquivo em windows-hadoop/bin antes de executar o notebook.")

DATA_PATH = PROJECT_ROOT / "data" / "vendas.csv"
DELTA_PATH = PROJECT_ROOT / "warehouse" / "delta" / "vendas_delta"

shutil.rmtree(DELTA_PATH, ignore_errors=True)
DELTA_PATH.parent.mkdir(parents=True, exist_ok=True)

builder = (
    SparkSession.builder
    .appName("delta-lake-demo")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "2")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

delta_uri = DELTA_PATH.as_posix()
delta_uri


In [ ]:
vendas_df = (
    spark.read
    .option("header", True)
    .csv(str(DATA_PATH))
    .withColumn("id_venda", F.col("id_venda").cast("bigint"))
    .withColumn("id_cliente", F.col("id_cliente").cast("int"))
    .withColumn("id_produto", F.col("id_produto").cast("int"))
    .withColumn("data_venda", F.to_date("data_venda"))
    .withColumn("quantidade", F.col("quantidade").cast("int"))
    .withColumn("preco_unitario", F.col("preco_unitario").cast("double"))
)

vendas_df.createOrReplaceTempView("staging_vendas")
vendas_df.orderBy("id_venda").show(truncate=False)


In [ ]:
spark.sql(
    f"""
    CREATE TABLE delta.`{delta_uri}`
    USING DELTA
    AS SELECT * FROM staging_vendas
    """
)

spark.sql(
    f"SELECT id_venda, nome_cliente, nome_produto, quantidade, status_pagamento FROM delta.`{delta_uri}` ORDER BY id_venda"
).show(truncate=False)


## INSERT em Delta Lake

A instrucao abaixo adiciona um novo registro na tabela Delta.


In [ ]:
spark.sql(
    f"""
    INSERT INTO delta.`{delta_uri}`
    VALUES (
        1011,
        6,
        'Marcos Lima',
        'Criciuma',
        106,
        'Mouse Gamer',
        'Perifericos',
        DATE '2026-03-11',
        2,
        180.00,
        'pago'
    )
    """
)

spark.sql(f"SELECT * FROM delta.`{delta_uri}` WHERE id_venda = 1011").show(truncate=False)


## UPDATE em Delta Lake

Agora o status e a quantidade de uma venda sao alterados diretamente na tabela.


In [ ]:
spark.sql(
    f"""
    UPDATE delta.`{delta_uri}`
    SET status_pagamento = 'estornado', quantidade = 3
    WHERE id_venda = 1003
    """
)

spark.sql(f"SELECT * FROM delta.`{delta_uri}` WHERE id_venda = 1003").show(truncate=False)


## DELETE em Delta Lake

Por fim, um registro cancelado e removido da tabela.


In [ ]:
spark.sql(
    f"""
    DELETE FROM delta.`{delta_uri}`
    WHERE id_venda = 1008
    """
)

spark.sql(f"SELECT * FROM delta.`{delta_uri}` WHERE id_venda = 1008").show(truncate=False)


In [ ]:
spark.sql(
    f"SELECT id_venda, nome_cliente, nome_produto, quantidade, status_pagamento FROM delta.`{delta_uri}` ORDER BY id_venda"
).show(truncate=False)

spark.sql(f"DESCRIBE HISTORY delta.`{delta_uri}`").show(truncate=False)


In [ ]:
spark.stop()
